In [ ]:
!python3 -m pip install kagglehub


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix,classification_report,accuracy_score


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    file_path,
)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.dropna()

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df = df.drop(columns=["customerID"])

categorical_cols = df.select_dtypes(include=["object"]).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df.drop("Churn", axis=1)
y = df["Churn"]


In [ ]:
df["Churn"].value_counts(normalize=True)

sns.countplot(x="Churn", data=df)
plt.show()

sns.histplot(df["tenure"], bins=30, kde=True)
plt.show()

sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.show()

contract_cols = [col for col in df.columns if "Contract_" in col]
churn_by_contract = df.groupby("Churn")[contract_cols].mean()
churn_by_contract

corr = df.corr()
top_corr = corr["Churn"].sort_values(ascending=False).head(10)
top_corr


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_val_scaled)


In [ ]:
print("Baseline Model")
print(classification_report(y_val, y_pred_baseline))


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)


In [ ]:
print("Improved Model")
print(classification_report(y_val, y_pred_rf))

In [ ]:
baseline_cm = confusion_matrix(y_val, y_pred_baseline)
rf_cm = confusion_matrix(y_val, y_pred_rf)

plt.figure(figsize=(5, 4))
sns.heatmap(baseline_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

plt.figure(figsize=(5, 4))
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Greens")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
print("Logistic Regression Classification Report")
print(classification_report(y_val, y_pred_baseline))

print("Random Forest Classification Report")
print(classification_report(y_val, y_pred_rf))


In [ ]:
baseline_accuracy = accuracy_score(y_val, y_pred_baseline)
rf_accuracy = accuracy_score(y_val, y_pred_rf)

baseline_accuracy, rf_accuracy


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_val)


In [ ]:
xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
xgb_accuracy


In [ ]:
xgb_cm = confusion_matrix(y_val, y_pred_xgb)

sns.heatmap(xgb_cm, annot=True, fmt="d")
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
